# Recent Results Overview

This notebook focuses on the most recent probe result group under `results/`, with Pearson correlation as the primary evaluation metric.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("ggplot")
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent

RESULTS_DIR = ROOT / "results"

print(f"Project root: {ROOT}")
print(f"Results dir:  {RESULTS_DIR}")

In [ ]:
def parse_run_path(path: Path) -> dict:
    rel = path.relative_to(RESULTS_DIR)
    if len(rel.parts) < 9:
        raise ValueError(f"Unexpected results path: {rel}")

    tail = rel.parts[-9:]
    prefix = rel.parts[:-9]
    probe_name, model_name, encoding, control_task, fold, seed, run_id, status, filename = tail

    layer_match = re.search(r"(?:_L|__L)(\d{3})", probe_name)
    if layer_match is None:
        raise ValueError(f"Could not parse layer from {probe_name}")

    return {
        "results_group": "/".join(prefix),
        "probe_name": probe_name,
        "layer": int(layer_match.group(1)),
        "model_name": model_name,
        "encoding": encoding,
        "control_task": control_task,
        "fold": int(fold),
        "seed": int(seed),
        "run_id": int(run_id),
        "status": status,
        "filename": filename,
        "path": path,
        "mtime": path.stat().st_mtime,
    }


def infer_metadata_path(results_group: str) -> Path | None:
    if "eval_adoption_absolute_accuracy_decay_aggregated" in results_group:
        candidate = ROOT / "data" / "eval_adoption_internals_aggregated" / "metadata.csv"
    elif "eval_adoption_absolute_accuracy_decay_table" in results_group:
        candidate = ROOT / "data" / "eval_adoption_internals_table" / "metadata.csv"
    else:
        candidate = ROOT / "data" / "internals" / "metadata.csv"
    return candidate if candidate.exists() else None


def last_non_null(series: pd.Series):
    cleaned = series.dropna()
    return cleaned.iloc[-1] if not cleaned.empty else np.nan


def maybe_metric(df: pd.DataFrame, column: str, reducer: str = "last"):
    if column not in df.columns:
        return np.nan
    series = df[column]
    if reducer == "last":
        return last_non_null(series)
    if reducer == "max":
        return series.max(skipna=True)
    if reducer == "min":
        return series.min(skipna=True)
    raise ValueError(reducer)


def load_metrics() -> tuple[pd.DataFrame, pd.DataFrame]:
    run_rows = []
    curve_frames = []

    for metrics_path in sorted(RESULTS_DIR.rglob("metrics.csv")):
        info = parse_run_path(metrics_path)
        df = pd.read_csv(metrics_path)

        curve = df.copy()
        for key, value in info.items():
            if key not in {"path", "filename"}:
                curve[key] = value
        curve_frames.append(curve)

        run_rows.append({
            **{k: v for k, v in info.items() if k not in {"path", "filename"}},
            "metrics_path": str(metrics_path),
            "epochs_logged": len(df),
            "best_val_loss": maybe_metric(df, "val loss", "min"),
            "best_val_error": maybe_metric(df, "val error", "min"),
            "best_val_pearson": maybe_metric(df, "val pearson", "max"),
            "full_test_error": maybe_metric(df, "full test error", "last"),
            "full_test_pearson": maybe_metric(df, "full test pearson", "last"),
            "unseen_test_error": maybe_metric(df, "unseen test error", "last"),
            "unseen_test_pearson": maybe_metric(df, "unseen test pearson", "last"),
        })

    runs_df = pd.DataFrame(run_rows).sort_values(["results_group", "layer", "control_task"]).reset_index(drop=True)
    curves_df = pd.concat(curve_frames, ignore_index=True).sort_values(["results_group", "layer", "control_task", "step"])
    return runs_df, curves_df


def load_predictions() -> pd.DataFrame:
    pred_frames = []

    for preds_path in sorted(RESULTS_DIR.rglob("preds.csv")):
        info = parse_run_path(preds_path)
        df = pd.read_csv(preds_path)
        df = df.rename(columns={df.columns[0]: "row_idx"})
        df["problem_id"] = df["instance"].str.extract(r"problem_(\d+)")[0].astype("string")
        df["row_id"] = df["instance"].str.extract(r"row_(\d+)")[0].astype("string")
        df["pred_abs_error"] = (df["pred"] - df["label"]).abs()
        df["pred_is_exact"] = np.isclose(df["pred"], df["label"]).astype(int)
        for key, value in info.items():
            if key not in {"path", "filename"}:
                df[key] = value

        metadata_path = infer_metadata_path(info["results_group"])
        if metadata_path is not None:
            metadata = pd.read_csv(metadata_path)
            merge_key = "row_id" if "row_id" in metadata.columns and df["row_id"].notna().any() else "problem_id"
            if merge_key in metadata.columns:
                metadata = metadata.copy()
                metadata[merge_key] = metadata[merge_key].astype("string")
                df[merge_key] = df[merge_key].astype("string")
                df = df.merge(metadata, on=merge_key, how="left", suffixes=("", "_meta"))

        pred_frames.append(df)

    return pd.concat(pred_frames, ignore_index=True).sort_values(["results_group", "layer", "control_task"]).reset_index(drop=True)


runs_df, curves_df = load_metrics()
preds_df = load_predictions()

In [ ]:
latest_group = (
    runs_df.groupby("results_group")["mtime"]
    .max()
    .sort_values(ascending=False)
    .index[0]
)

latest_runs = runs_df[runs_df["results_group"] == latest_group].copy()
latest_curves = curves_df[curves_df["results_group"] == latest_group].copy()
latest_preds = preds_df[preds_df["results_group"] == latest_group].copy()

pd.DataFrame({
    "latest_results_group": [latest_group],
    "n_runs": [len(latest_runs)],
    "n_prediction_rows": [len(latest_preds)],
    "controls": [", ".join(sorted(latest_runs["control_task"].unique()))],
    "layers": [latest_runs["layer"].nunique()],
})

In [ ]:
pearson_summary = latest_runs.groupby("control_task").agg(
    best_full_test_pearson=("full_test_pearson", "max"),
    best_unseen_test_pearson=("unseen_test_pearson", "max"),
    mean_full_test_pearson=("full_test_pearson", "mean"),
    mean_unseen_test_pearson=("unseen_test_pearson", "mean"),
    min_full_test_error=("full_test_error", "min"),
    min_unseen_test_error=("unseen_test_error", "min"),
).round(4)
pearson_summary.sort_values("best_full_test_pearson", ascending=False)

In [ ]:
best_layers = (
    latest_runs.sort_values(["control_task", "full_test_pearson"], ascending=[True, False])
    .groupby("control_task")
    .head(1)
    [["control_task", "layer", "full_test_pearson", "unseen_test_pearson", "full_test_error", "unseen_test_error"]]
    .sort_values("full_test_pearson", ascending=False)
)
best_layers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

for control_task, group in latest_runs.groupby("control_task"):
    group = group.sort_values("layer")
    axes[0].plot(group["layer"], group["full_test_pearson"], marker="o", linewidth=2, label=control_task)
    axes[1].plot(group["layer"], group["unseen_test_pearson"], marker="o", linewidth=2, label=control_task)

axes[0].set_title("Full-Test Pearson by Layer")
axes[0].set_xlabel("Layer")
axes[0].set_ylabel("Pearson correlation")
axes[1].set_title("Unseen-Test Pearson by Layer")
axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Pearson correlation")
axes[0].legend(title="Control")
plt.tight_layout()

In [ ]:
heatmap_df = latest_runs.pivot(index="control_task", columns="layer", values="full_test_pearson").sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(heatmap_df.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-1, vmax=1)
ax.set_title("Full-Test Pearson Heatmap")
ax.set_xlabel("Layer")
ax.set_ylabel("Control")
ax.set_xticks(np.arange(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns)
ax.set_yticks(np.arange(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index)
plt.colorbar(im, ax=ax, label="Pearson correlation")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for control_task, group in latest_runs.groupby("control_task"):
    group = group.sort_values("layer")
    ax.plot(group["layer"], group["full_test_error"], marker="o", linewidth=2, label=control_task)

ax.set_title("Supporting View: Full-Test Error by Layer")
ax.set_xlabel("Layer")
ax.set_ylabel("Error")
ax.legend(title="Control")
plt.tight_layout()

In [ ]:
top_controls = best_layers["control_task"].tolist()[:4]
fig, axes = plt.subplots(len(top_controls), 1, figsize=(8, 4 * max(1, len(top_controls))))
axes = np.atleast_1d(axes)

for ax, control_task in zip(axes, top_controls):
    best_layer = int(best_layers.loc[best_layers["control_task"] == control_task, "layer"].iloc[0])
    subset = latest_preds[(latest_preds["control_task"] == control_task) & (latest_preds["layer"] == best_layer)]
    ax.scatter(subset["label"], subset["pred"], alpha=0.7)
    if len(subset) > 1:
        lo = min(subset["label"].min(), subset["pred"].min())
        hi = max(subset["label"].max(), subset["pred"].max())
        ax.plot([lo, hi], [lo, hi], linestyle="--", color="black", linewidth=1)
    ax.set_title(f"{control_task} | best layer L{best_layer:03d}")
    ax.set_xlabel("True label")
    ax.set_ylabel("Prediction")

plt.tight_layout()

In [ ]:
latest_runs.sort_values("full_test_pearson", ascending=False)[[
    "control_task",
    "layer",
    "full_test_pearson",
    "unseen_test_pearson",
    "full_test_error",
    "unseen_test_error",
    "best_val_pearson",
    "best_val_error",
]].head(20)